In [0]:
output_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "bronze/bundestag/activities/year=2020/month=11/"
)



In [0]:
df = (
    spark.read
    .option("multiLine", "true")
    .json(output_path)
)


In [0]:
from pyspark.sql.functions import explode

activities = (
    df
    .select(explode("documents").alias("activity"))
    .select("activity.*")
)


In [0]:
from pyspark.sql import functions as F

print("Rows:", activities.count())
print("Columns:", len(activities.columns))

activities.printSchema()

activities.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in activities.columns
]).show(truncate=False)

Rows: 12934
Columns: 14
root
 |-- abstract: string (nullable = true)
 |-- aktivitaetsart: string (nullable = true)
 |-- aktualisiert: string (nullable = true)
 |-- datum: string (nullable = true)
 |-- deskriptor: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- typ: string (nullable = true)
 |-- dokumentart: string (nullable = true)
 |-- fundstelle: struct (nullable = true)
 |    |-- anfangsquadrant: string (nullable = true)
 |    |-- anfangsseite: long (nullable = true)
 |    |-- datum: string (nullable = true)
 |    |-- dokumentart: string (nullable = true)
 |    |-- dokumentnummer: string (nullable = true)
 |    |-- drucksachetyp: string (nullable = true)
 |    |-- endquadrant: string (nullable = true)
 |    |-- endseite: long (nullable = true)
 |    |-- frage_nummer: string (nullable = true)
 |    |-- herausgeber: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- pdf_url: str

In [0]:
activities_clean = activities.select(
    "id",
    "person_id",
    "wahlperiode",
    "datum",
    "aktualisiert",
    "aktivitaetsart",
    "dokumentart",
    "titel",

    F.col("fundstelle.id").alias("fundstelle_id"),
    F.col("fundstelle.dokumentnummer").alias("dokumentnummer"),
    F.col("fundstelle.drucksachetyp").alias("drucksachetyp"),
    F.col("fundstelle.herausgeber").alias("herausgeber"),
    F.col("fundstelle.datum").alias("fundstelle_datum"),
    F.col("fundstelle.verteildatum").alias("verteildatum"),
    F.col("fundstelle.pdf_url").alias("pdf_url"),
    F.col("fundstelle.frage_nummer").alias("frage_nummer"),
    F.col("fundstelle.urheber").alias("urheber"),

    "vorgangsbezug"
)

In [0]:
activities_clean = (
    activities_clean
    .withColumn("datum", F.to_date("datum"))
    .withColumn("fundstelle_datum", F.to_date("fundstelle_datum"))
    .withColumn("verteildatum", F.to_date("verteildatum"))
    .withColumn("aktualisiert", F.to_timestamp("aktualisiert"))
)

In [0]:
activities_clean.printSchema()
activities_clean.show(5, truncate=False)

root
 |-- id: string (nullable = true)
 |-- person_id: string (nullable = true)
 |-- wahlperiode: long (nullable = true)
 |-- datum: date (nullable = true)
 |-- aktualisiert: timestamp (nullable = true)
 |-- aktivitaetsart: string (nullable = true)
 |-- dokumentart: string (nullable = true)
 |-- titel: string (nullable = true)
 |-- fundstelle_id: string (nullable = true)
 |-- dokumentnummer: string (nullable = true)
 |-- drucksachetyp: string (nullable = true)
 |-- herausgeber: string (nullable = true)
 |-- fundstelle_datum: date (nullable = true)
 |-- verteildatum: date (nullable = true)
 |-- pdf_url: string (nullable = true)
 |-- frage_nummer: string (nullable = true)
 |-- urheber: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- vorgangsbezug: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- titel: string (nullable = true)
 |    |    |-- vorgangsposition: string (nullable = 

In [0]:
activity_descriptors = (
    activities_clean
    .select(
        F.col("id").alias("activity_id"),
        F.explode("deskriptor").alias("descriptor")
    )
    .select(
        "activity_id",
        F.col("descriptor.name").alias("descriptor_name"),
        F.col("descriptor.typ").alias("descriptor_type")
    )
)

activity_descriptors.show(20, truncate=False)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6448785485194790>, line 14
      1 activity_descriptors = (
      2     activities_clean
      3     .select(
   (...)
     11     )
     12 )
---> 14 activity_descriptors.show(20, truncate=False)

File <command-6448785485194790>, line 2
      1 activity_descriptors = (
----> 2     activities_clean
      3     .select(
      4         F.col("id").alias("activity_id"),
      5         F.explode("deskriptor").alias("descriptor")
      6     )
      7     .select(
      8         "activity_id",
      9         F.col("descriptor.name").alias("descriptor_name"),
     10         F.col("descriptor.typ").alias("descriptor_type")
     11     )
     12 )
     14 activity_descriptors.show(20, truncate=False)

File <command-6470786029171985>, line 2
      1 activities_clean = (
----> 2     activities_clean
      3     .withColumn("dat

In [0]:
activity_procedures = (
    activities_clean
    .select(
        F.col("id").alias("activity_id"),
        F.explode("vorgangsbezug").alias("procedure")
    )
    .select(
        "activity_id",
        F.col("procedure.id").alias("procedure_id"),
        F.col("procedure.titel").alias("procedure_title"),
        F.col("procedure.vorgangsposition").alias("procedure_position"),
        F.col("procedure.vorgangstyp").alias("procedure_type")
    )
)

activity_procedures.show(20, truncate=False)

+-----------+------------+----------------------------------------------------------------------------------------------------------------------------------+-------------------------------+--------------+
|activity_id|procedure_id|procedure_title                                                                                                                   |procedure_position             |procedure_type|
+-----------+------------+----------------------------------------------------------------------------------------------------------------------------------+-------------------------------+--------------+
|1489486    |269878      |Verbot von Grünlandumbruch streichen                                                                                              |Beschlussempfehlung und Bericht|Antrag        |
|1489485    |269878      |Verbot von Grünlandumbruch streichen                                                                                              |Beschlussempfehlung und

In [0]:
activities_clean = activities_clean.drop(
    "deskriptor",
    "vorgangsbezug"
)

In [0]:
jahr=2020
mo="11"
adls_path_activities_clean = (
    f"abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    f"silver/bundestag/activities/year={jahr}/month={mo}/"
)

adls_path_activity_descriptors = (
    f"abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    f"silver/bundestag/activity_descriptors/year={jahr}/month={mo}/"
)

adls_path_activity_procedures = (
    f"abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    f"silver/bundestag/activity_procedures/year={jahr}/month={mo}/"
)
activities_clean.coalesce(1).write.mode("overwrite").parquet(
    adls_path_activities_clean
)
"""
activity_descriptors.coalesce(1).write.mode("overwrite").parquet(
    adls_path_activity_descriptors
)
"""
activity_procedures.coalesce(1).write.mode("overwrite").parquet(
    adls_path_activity_procedures
)